# Lab 2 — Logistic Regression (Loan Default)

**Day 03 · Classification & Model Interpretation · Cisco AI/ML Training**

---

## Learning objectives

1. Prepare feature matrix $X$ and binary target $y$ for classification.
2. Split data with **stratification** to preserve class balance.
3. Fit **`LogisticRegression`** and interpret intercept / coefficients.
4. Compare **`predict_proba`** (probability) vs **`predict`** (hard label).

> **Checkpoints:** train **800** / test **200** · `int_rate` coef positive · sample preds include **0** and **1**

<!-- cisco-day03-lab02-expanded-2026 -->

**Day 3 flow:** [Lab 1 — Probability](lab01_probability_exercises.ipynb) → **Lab 2 (you are here) — Logistic regression** → [Lab 3 — Confusion matrix](lab03_confusion_matrix.ipynb) → [Lab 4 — ROC & AUC](lab04_roc_auc.ipynb) → [Lab 5 — sklearn Pipeline](lab05_sklearn_pipeline.ipynb) → [Lab 6 — SHAP](lab06_shap_interpretability.ipynb)

## Logistic regression in one slide

**Linear model on log-odds:**

$$\text{logit}(p) = \ln\frac{p}{1-p} = \beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p$$

**Sigmoid converts back to probability:**

$$p = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \cdots)}}$$

| vs Day 2 Linear Regression | Logistic Regression |
|---------------------------|---------------------|
| Target: continuous rating | Target: 0/1 default |
| Output: any real number | Output: probability in [0, 1] |
| `LinearRegression` | `LogisticRegression` |

---

## 1. Load data and select features

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif (GH_ROOT.parent / "notebooks").is_dir() and (GH_ROOT.parents[1] / "requirements-student.txt").is_file():
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "requirements-student.txt").is_file():
            GH_ROOT = parent
            break

LENDING_CLUB_CSV = GH_ROOT / "data" / "lending-club" / "lending_club_sample.csv"
DEFAULT_STATUSES = {"Charged Off", "Late (31-120 days)"}
NUMERIC_FEATURES = ["loan_amnt", "int_rate", "annual_inc", "dti", "installment"]
CATEGORICAL_FEATURES = ["grade", "term"]

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

df = pd.read_csv(LENDING_CLUB_CSV)
df["default"] = df["loan_status"].isin(DEFAULT_STATUSES).astype(int)

X = df[NUMERIC_FEATURES]
y = df["default"]

print(f"X shape: {X.shape}")
print(f"default rate: {y.mean():.4f}")
display(X.head(3))


### 1b. Feature dtypes and ranges

In [ ]:
display(X.describe().round(2))
print("Any missing?", X.isna().any().any())


### 1c. Correlation among numeric features

In [ ]:
corr = X.corr().round(3)
display(corr)


---

## 2. Train/test split with stratification

`stratify=y` keeps the same default rate in train and test — important when classes are imbalanced (critical on Day 6).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"train size: {len(X_train)}")
print(f"test size:  {len(X_test)}")
print(f"default rate (train): {y_train.mean():.4f}")
print(f"default rate (test):  {y_test.mean():.4f}")

assert len(X_train) == 800 and len(X_test) == 200


### 2b. Why 80/20 and random_state=42

In [ ]:
print("800 train rows → fit coefficients")
print("200 test rows  → honest evaluation in Labs 3–4")
print("random_state=42 → same split every time you re-run")


---

## 3. Fit logistic regression

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

print("Lab 2 — Logistic regression")
print(f"intercept (beta_0): {model.intercept_[0]:.4f}")
for name, coef in zip(NUMERIC_FEATURES, model.coef_[0]):
    print(f"  {name}: {coef:.4f}")


### Coefficient interpretation

- **`int_rate` (+):** Higher interest rate → higher log-odds of default.
- **`dti` (+):** Higher debt-to-income → higher default log-odds.
- Coefficients are on the **log-odds scale**, not probability directly.

### 3b. Rank features by |coefficient|

In [ ]:
coef_df = pd.DataFrame({
    "feature": NUMERIC_FEATURES,
    "coef": model.coef_[0],
    "abs_coef": np.abs(model.coef_[0]),
}).sort_values("abs_coef", ascending=False)
display(coef_df.round(4))


---

## 4. Sigmoid visualization

In [ ]:
x = np.linspace(-6, 6, 200)
sigmoid = 1 / (1 + np.exp(-x))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x, sigmoid, color="steelblue", lw=2)
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.7)
ax.axvline(0, color="gray", linestyle="--", alpha=0.7)
ax.set_xlabel("log-odds (eta)")
ax.set_ylabel("probability p")
ax.set_title("Sigmoid: log-odds → probability")
plt.tight_layout()
plt.show()


At log-odds = 0, probability = **0.5** — the default classification threshold for `predict()`.

---

## 5. `predict_proba` vs `predict`

| Method | Output |
|--------|--------|
| `predict_proba(X)[:, 1]` | P(default=1) per row |
| `predict(X)` | 0 or 1 using threshold **0.5** |

In [ ]:
proba = model.predict_proba(X_test.head(3))[:, 1]
pred = model.predict(X_test.head(3))

comparison = pd.DataFrame({
    "int_rate": X_test.head(3)["int_rate"].values,
    "dti": X_test.head(3)["dti"].values,
    "P(default)": proba.round(4),
    "predicted_label": pred,
    "actual_default": y_test.head(3).values,
})
display(comparison)

print(f"sample P(default): {proba.round(4)}")
print(f"sample predictions: {pred.tolist()}")
assert 0 in pred and 1 in pred


Row with P(default) ≥ **0.5** → predicted **1**. Below 0.5 → predicted **0**.

### 5b. Probability distribution on test set

In [ ]:
test_proba = model.predict_proba(X_test)[:, 1]
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(test_proba[y_test == 0], bins=20, alpha=0.6, label="actual 0", density=True)
ax.hist(test_proba[y_test == 1], bins=20, alpha=0.6, label="actual 1", density=True)
ax.set_xlabel("P(default)")
ax.legend()
ax.set_title("Score separation — modest overlap")
plt.tight_layout()
plt.show()


---

## 6. Manual log-odds for one test row

In [ ]:
row = X_test.iloc[0]
eta = model.intercept_[0] + np.dot(model.coef_[0], row.values)
p_manual = 1 / (1 + np.exp(-eta))
p_sklearn = model.predict_proba(row.values.reshape(1, -1))[0, 1]

print(f"log-odds eta: {eta:.4f}")
print(f"manual p:     {p_manual:.4f}")
print(f"sklearn p:    {p_sklearn:.4f}")


### 6b. Break down eta by feature

In [ ]:
contributions = model.coef_[0] * row.values
contrib_df = pd.DataFrame({
    "feature": NUMERIC_FEATURES,
    "value": row.values,
    "beta*x": contributions,
})
display(contrib_df.round(4))
print(f"intercept + sum(beta*x) = {model.intercept_[0] + contributions.sum():.4f}")


---

## 7. Exploratory plots — features vs default

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(data=df, x="default", y="int_rate", ax=axes[0], palette="Set2")
axes[0].set_title("Interest rate by default")
sns.boxplot(data=df, x="default", y="dti", ax=axes[1], palette="Set2")
axes[1].set_title("DTI by default")
plt.tight_layout()
plt.show()


Defaults tend toward higher `int_rate` and `dti` — consistent with positive coefficients.

### 7b. Annual income and loan amount

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(data=df, x="default", y="annual_inc", ax=axes[0], palette="Set2")
axes[0].set_title("Annual income by default")
sns.boxplot(data=df, x="default", y="loan_amnt", ax=axes[1], palette="Set2")
axes[1].set_title("Loan amount by default")
plt.tight_layout()
plt.show()


---

## 8. Training accuracy (not the final metric)

In [ ]:
from sklearn.metrics import accuracy_score

train_acc = accuracy_score(y_train, model.predict(X_train))
test_acc = accuracy_score(y_test, model.predict(X_test))
print(f"train accuracy: {train_acc:.4f}")
print(f"test accuracy:  {test_acc:.4f}")
print("(Lab 3 adds confusion matrix, precision, recall, F1)")


### 8b. Overfitting check

In [ ]:
gap = train_acc - test_acc
print(f"train - test gap: {gap:.4f}")
print("Small gap here — model is simple; still evaluate on held-out test.")


### 8c. Default rate by predicted probability bucket

In [ ]:
buckets = pd.cut(test_proba, bins=[0, 0.3, 0.5, 0.7, 1.0])
cal = pd.DataFrame({"bucket": buckets, "actual": y_test.values}).groupby("bucket")["actual"].mean()
print("Empirical default rate by score bucket:")
print(cal.round(3))


---

## 8d. Standardize features (preview of Lab 6)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model_s = LogisticRegression(max_iter=1000, random_state=42)
model_s.fit(X_train_s, y_train)
print("Scaled model test accuracy:", round(accuracy_score(y_test, model_s.predict(X_test_s)), 4))
print("(Coefficients change scale; predictions at 0.5 threshold stay similar.)")


### 8e. Installment vs default

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
sns.boxplot(data=df, x="default", y="installment", ax=ax, palette="Set2")
ax.set_title("Monthly installment by default status")
plt.tight_layout()
plt.show()


### 8f. Grade distribution in training set

In [ ]:
print(df.loc[X_train.index, "grade"].value_counts().sort_index())


### 8g. Predict on a synthetic high-risk profile

In [ ]:
high_risk = pd.DataFrame([{
    "loan_amnt": 25000, "int_rate": 18.0, "annual_inc": 45000,
    "dti": 28.0, "installment": 850,
}])
p_hr = model.predict_proba(high_risk)[0, 1]
print(f"P(default) high-risk profile: {p_hr:.4f}")


### 8h. Predict on a synthetic low-risk profile

In [ ]:
low_risk = pd.DataFrame([{
    "loan_amnt": 10000, "int_rate": 7.5, "annual_inc": 95000,
    "dti": 8.0, "installment": 310,
}])
p_lr = model.predict_proba(low_risk)[0, 1]
print(f"P(default) low-risk profile: {p_lr:.4f}")


---

## 9. Try it yourself

Pick one test row. Compute log-odds manually and compare to `predict_proba`.

In [ ]:
# Your code here


In [ ]:
row2 = X_test.iloc[5]
eta2 = model.intercept_[0] + np.dot(model.coef_[0], row2.values)
p2 = 1 / (1 + np.exp(-eta2))
print(f"manual P(default): {p2:.4f}")
print(f"sklearn:           {model.predict_proba(row2.values.reshape(1,-1))[0,1]:.4f}")


---

## 10. Checkpoint summary

In [ ]:
print("=" * 50)
print(f"train: {len(X_train)}, test: {len(X_test)}")
print(f"intercept: {model.intercept_[0]:.4f}")
print(f"int_rate coef: {model.coef_[0][1]:.4f} (expect > 0)")
assert model.coef_[0][1] > 0
assert model.coef_[0][3] > 0  # dti
print("\n✓ Checkpoint assertions passed")


---

## Reflection questions

1. What happens to predictions if you change the threshold from 0.5 to 0.7?
2. Why use `stratify=y` even when classes are nearly balanced?
3. Which metrics evaluate this model best? *(Lab 3 — confusion matrix, Lab 4 — AUC)*
4. Why are coefficients on log-odds, not probability?

**Previous:** [Lab 1 — Probability](lab01_probability_exercises.ipynb)  
**Next:** [Lab 3 — Confusion matrix](lab03_confusion_matrix.ipynb)